In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (benchmarks still run):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
adapter_dirs = {}
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    v = variant.split("_")[0]
    cands = sorted(glob.glob(f"/kaggle/input/**/{v}/adapter", recursive=True))
    cands += sorted(glob.glob("/kaggle/input/arc-tier-b-adapters/{v}/adapter"))
    hits = [c for c in cands if (Path(c) / "adapter_config.json").exists()]
    if hits:
        adapter_dirs[variant] = hits[0]
        print(f"adapter [{variant}] ->{hits[0]}")
    else:
        print(f"!! NO adapter output for [{variant}]")
if not adapter_dirs:
    print("INPUT DIRS:", os.listdir("/kaggle/input"))
    for root in (Path("/kaggle/input")).rglob("adapter_config.json"):
        print("found adapter_config:", root)
    raise SystemExit("No adapters found - is dataset niyuvo/arc-tier-b-adapters attached?")
open("/kaggle/working/adapters.json", "w").write(json.dumps(adapter_dirs, indent=2))


In [ ]:
import json, os, subprocess, sys
adapter_dirs = json.load(open("/kaggle/working/adapters.json"))
CONFIGS = [["spreadA", {"bias": 0.5, "k": 10, "halt_threshold": 0.42, "min_gain": 0.035}], ["spreadB", {"bias": 0.48, "k": 10, "halt_threshold": 0.4, "min_gain": 0.04}], ["spreadC", {"bias": 0.45, "k": 12, "halt_threshold": 0.38, "min_gain": 0.05}], ["spreadE", {"bias": 0.52, "k": 8, "halt_threshold": 0.4, "min_gain": 0.03}], ["hard2b", {"bias": 0.38, "k": 12, "halt_threshold": 0.33, "min_gain": 0.07}], ["default", {"bias": 0.6, "k": 12, "halt_threshold": 0.45, "min_gain": 0.02}]]
HEADROOM = ["headroomE8", {"bias": 0.52, "k": 8, "halt_threshold": 0.4, "min_gain": 0.03}]
tasks = "arc_easy,arc_challenge,hellaswag,piqa,winogrande,boolq,sciq"
limits = "arc_easy=40,arc_challenge=40,hellaswag=40,piqa=40,winogrande=40,boolq=40,sciq=40"
os.makedirs("/kaggle/working/spread", exist_ok=True)
# fixed-depth reference curve (budgeted caps): max_loops = 1, 2, 3
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    for ml in (1, 2, 3):
        out = f"/kaggle/working/spread/spread-{variant}-fixed{ml}.csv"
        cmd = [sys.executable, "scripts/run_benchmarks.py",
               "--base", "/kaggle/working/jetmoe-8b",
               "--model", variant,
               "--adapters", f"{variant}={adapter_dirs[variant]}",
               "--budgeted", "--max_loops", str(ml),
               "--tasks", tasks, "--limits", limits,
               "--out", out]
        print(">>>", variant, "fixed", ml)
        subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)
# adaptive configs at max_loops=4
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    for cname, ckw in CONFIGS:
        out = f"/kaggle/working/spread/spread-{variant}-{cname}.csv"
        cmd = [sys.executable, "scripts/run_benchmarks.py",
               "--base", "/kaggle/working/jetmoe-8b",
               "--model", variant,
               "--adapters", f"{variant}={adapter_dirs[variant]}",
               "--budgeted", "--max_loops", "4",
               "--tasks", tasks, "--limits", limits,
               "--controller", json.dumps(ckw),
               "--out", out]
        print(">>>", variant, cname, json.dumps(ckw))
        subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)
# headroom: same as spreadE but max_loops=8
hname, hkw = HEADROOM
for variant in ['model_adaptive', 'block_adaptive', 'layer_adaptive']:
    out = f"/kaggle/working/spread/spread-{variant}-{hname}.csv"
    cmd = [sys.executable, "scripts/run_benchmarks.py",
           "--base", "/kaggle/working/jetmoe-8b",
           "--model", variant,
           "--adapters", f"{variant}={adapter_dirs[variant]}",
           "--budgeted", "--max_loops", "8",
           "--tasks", tasks, "--limits", limits,
           "--controller", json.dumps(hkw),
           "--out", out]
    print(">>>", variant, hname, json.dumps(hkw))
    subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)


In [ ]:
import csv, glob, math, os, shutil, json
os.makedirs("/kaggle/output/spread", exist_ok=True)
csvs = sorted(glob.glob("/kaggle/working/spread/spread-*.csv"))
for c in csvs:
    shutil.copy(c, f"/kaggle/output/spread/{os.path.basename(c)}")
print(len(csvs), "spread CSVs saved")
mcq = ["arc_easy", "arc_challenge", "hellaswag", "piqa", "winogrande", "boolq", "sciq"]
def tag_of(fn):
    return os.path.basename(fn)[len("spread-"):].replace(".csv", "")
for key in ["model_adaptive", "block_adaptive", "layer_adaptive"]:
    files = sorted(glob.glob(f"/kaggle/working/spread/spread-{key}-*.csv"))
    print("\n===", key, "===")
    print(f"{'config':11s} {'acc':>5s} {'loops':>5s} {'GFLOP':>6s} {'acc/G':>5s} {'nan':>3s}  hist")
    rowsacc = {}
    for c in files:
        rows = list(csv.DictReader(open(c)))
        accs = [float(r["acc"]) * 100 for r in rows if r["task"] in mcq]
        avg = sum(accs) / len(accs)
        gflop = sum(float(r.get("avg_flops_per_item", 0)) for r in rows if r["task"] in mcq) / len(accs) / 1e9
        r0 = rows[0]
        hist = r0.get("halt_hist", "")
        print(f"{tag_of(c):11s} {avg:5.1f} {r0['avg_loops_per_item']:>5.1f} {gflop:6.1f} {avg/gflop:5.2f} {r0['nan_halts']:>3}  {hist}")
        rowsacc[tag_of(c)] = avg
    print("-> BEST:", best_is := max(rowsacc, key=rowsacc.get), f"{rowsacc[best_is]:.1f}%")
